<img src="uva_seal.png">  

## Apache Kafka Introduction

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: October 4, 2026

---  


### SOURCES

https://kafka.apache.org/intro

https://www.conduktor.io/glossary/kafka-consumer-groups-explained

### OBJECTIVES
- Understand the benefits of storing data in logs instead of databases
- Understand the use cases for Kafka
- Differentiate between Producers and Consumers
- Understand the benefits of decoupling Producers from Consumers, and the Pub/Sub model
- Understand the benefits of Topic Partitioning
- Identify how Kafka provides Scalability and Durability
- Explain the tradeoffs of the three delivery models
- Understand the differences between Kafka and traditional message queues

### CONCEPTS

- Events
- Logs
- Event Streaming
- Topics
- Durability
- Producers
- Consumers
- Topic Partitioning
- Offsets
- Consumer Groups
- Keys and Ordering
- Retention and Replay
- Idempotent Operations

---

### 1. What is Apache Kafka?

Open-source, distributed event streaming platform for high-performance data pipelines, streaming analytics, data integration, and mission-critical applications.

As of July 2021, used by more than 80% of Fortune 100 companies.


---

### 2. Kafka Objects: Things and Events

Databases store **things** that have a **state**...cars, customers, ...

Many dynamic things can be more naturally thought of as **events** ... hospital admission, patient treatment, website session, ...

Hard to store events in databases...does each dose of a medication get a record? Maybe other pieces of information need to be added later? 

---

### 3. Logs

Instead, can store events in a `log`  
A log is ordered sequence of events, with some state info  
Logs are easy to think about, and build at scale

<img src="logfile_example.png">  

---

### 4. Event Streaming  

The practice of capturing data in real-time from event sources like `databases`, `sensors`, `mobile devices`, `cloud services`, and `software applications` in the form of streams of events 

- durably stored for later retrieval  

- manipulate, process, and react to the event streams in real-time as well as retrospectively  

- route the event streams to different destination technologies as needed (e.g., **Spark**, Tableau) 

---

### 5. What Kafka Does

`Kafka` is a system for managing logs 

Logs are organized into `topics`  

A topic is an ordered collection of events stored in **durable** way (written to disk and replicated)

Topics can be small or enormous

Example: **Slack**. Topics are slotted into *channels*, and users can join a channel


---

### 6. How Topics are Used

Write lots of small programs instead of large monolith  

Each service can talk to (consume) Kafka topics

Then produce the message to another Kafka topic

Now there's persistent data in streams

It is possible to build new services that use these streams

---

### 7. Publishing Models

#### Before Pub/Sub

The producers of content needed to track their consumers (who is signed up to receive Time magazine?)

Needs to be maintained over time, and scalability is a challenge

#### The Pub/Sub Model 

The Pub/Sub model uses Producers and Consumers

**Producers** are client applications that publish (write) events to various Kafka topics

**Consumers** are client applications that subscribe to (read and process) the events from topics of interest 

Producers and Consumers are fully decoupled and agnostic of each other...this allows the service to scale.  

Producers never need to wait for consumers.

**Pub/Sub Messaging**

<img src="pubsub.png">  

---

### 8. Topics are Partitioned

*How do things scale?*

A topic is spread over a number of "buckets" located on different Kafka brokers (aka nodes or servers). 

This distributed placement of data is important for scalability.  
Allows client applications to both read and write the data from/to many brokers at the same time. 

When new event is published to a topic, it is appended to one of the topic's partitions.  
Events with the same event key (e.g., a patient ID) are written to same partition, 

Kafka guarantees any consumer of a topic-partition will always read that partition's events in exactly the same order as they were written.

<img src="partitioned_topics.png" alt="drawing" width="800">  

---

### 9. Partitions are Replicated

For **fault-tolerance** and **high availability**, each topic is replicated.

The **leader** is the first broker that receives the data partition. The leader sends data to other available brokers, called **followers**.

The follower is also known as an **in-sync replica**. 

Replication can be across geographies, datacenters.  
In this way, multiple brokers keep copies of the data.

It is common to use replication factor = 3

---

### 10. Consumer Groups

A **consumer group** is a set of consumers that cooperate to process a topic.

* Each consumer in the group is assigned a subset of the topic's partitions
* No two consumers in the same group read from the same partition simultaneously
* Using multiple consumers allows a topic to be processed in parallel

Consider this example:

<img src="consumer_group.png" alt="drawing" width="400">  

In this example, each consumer is assigned two partitions. 

A rebalance trigger handles potential consumer 2 failure.

Additional rebalance triggers would handle situations including:

* A consumer joins the group
* A consumer leaves (gracefully or due to failure)
* A consumer is considered dead (missed heartbeat deadline)
* Partitions are added to a subscribed topic

---

### 11. Keys, Partitioning, and Ordering
When there is related data in Kafka, it is important to control how it's grouped and ordered.

**Example:**   
Consider a patient-data application

Want to ensure events for same patient remain ordered

Can use `patient_id` as key

**How is this implemented?**  

A producer can specify a **key** when sending an event.  

Kafka uses the key to determine which partition receives the event.
.

```text
key = patient_123
       ↓
   partitioner
       ↓
   Partition 2
```

Events with the same key are normally sent to the same partition.

This is important because Kafka guarantees ordering **within a partition**, but not across partitions.

```text
Partition 0: A → B → C
Partition 1: D → E → F
```

Kafka does **not** guarantee whether A occurred bethe scope of **ordering**.


---

### 12. Retention and Replay

Kafka stores events for a configured **retention period** rather than deleting an event as soon as it is consumed.

```text
Producer
   ↓
Kafka log
   ↓
E1 → E2 → E3 → E4 → E5
          ↑
      Consumer offset
```

A consumer keeps track of its position using an **offset**.

A new consumer can start from an earlier offset and **replay historical events**.

This makes it possible to:

- Reprocess data after fixing a bug
- Build a new application using historical events
- Recover from processing failures
- Have multiple applications independently process the same events


Note: The Producer is NOT responsible for maintaining Consumer offsets.

**Example**:  
Returning to Slack, the app will show the last message read in a channel.  
Your machine tracks this position.

<img src="kafka_offsets.png" alt="drawing" width="400">  

---

### 13. Delivery Semantics

Distributed systems must deal with failures while messages are being processed.

Distributed applications must explicitly decide how to handle this.

**Example:**

Event: "Charge customer $100"

Consumer:  
1. Receives event  
2. Charges customer $100  
3. Crashes before recording that it processed the event

Consumer restarts:  
4. Receives the same event again  
5. Charges customer another $100

This would be a problem!

---

**Three Delivery Models**  

Kafka applications therefore need to consider three delivery models:

| Model             | Meaning                                                   |
| ----------------- | --------------------------------------------------------- |
| **At-most-once**  | An event may be lost, but is not processed twice          |
| **At-least-once** | An event is not lost, but may be processed more than once |
| **Exactly-once**  | An event's effect is intended to occur only once          |

At-least-once processing is common, so applications often need **idempotent operations**.

An idempotent operation can be performed multiple times with the same result as performing it once.

For example:

```text
set_balance(account, $100)
```

is safer to repeat than:

```text
increase_balance(account, $100)
```


---

### 14. Kafka vs. Traditional Message Queues

Kafka is often compared with traditional message-queue systems.

| Traditional queue                                | Kafka                                |
| ------------------------------------------------ | ------------------------------------ |
| Messages are typically removed after consumption | Events are retained                  |
| Consumers compete for messages                   | Consumer groups divide partitions    |
| Replay is often difficult                        | Replay is a core feature             |
| Primarily focused on messaging                   | Durable event log + messaging        |
| Historical events are usually unavailable        | Historical events can be reprocessed |

For example, three applications can independently consume the same Kafka topic:

```text
                 ┌── Analytics
                 │
Events → Kafka ──┼── Fraud Detection
                 │
                 └── Data Warehouse
```

Each application maintains its own consumer-group offsets.
